<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook4_Two_Compartment_Model_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4: Two-Compartment Model and Drug Distribution

This notebook is an advanced module in the “fundamental pharmacokinetic models” section of the ClinPKPD interactive teaching platform.

In previous notebooks, you learned about:

- Dose, volume of distribution, clearance, and half-life in a one-compartment model
- Concentration–time profiles after intravenous and oral dosing
- Multiple dosing, accumulation, and steady state

The one-compartment model assumes that a drug rapidly distributes into one well-mixed space. This assumption is useful for building basic PK intuition, but some drugs show a rapid decline shortly after intravenous administration followed by a slower terminal decline. A single compartment and a single exponential process may not adequately describe the entire profile.

This notebook introduces the following idea:

> How does a drug distribute between a central and a peripheral compartment, and how is distribution different from true elimination from the body?

The intravenous bolus two-compartment model is used to illustrate:

- Central and peripheral compartments
- Elimination clearance $CL$
- Intercompartmental clearance $Q$
- Central volume $V_c$ and peripheral volume $V_p$
- A rapid distribution phase and a slower terminal phase
- The influence of sampling times on model identification

This module also prepares you for the later notebook on noncompartmental analysis (NCA), because terminal half-life and sampling design are easier to interpret after the distribution and terminal phases have been distinguished.

**Important note:**

This notebook is a teaching simulation, not a clinical dosing calculator for a specific drug or patient. The example parameters are educational and are not recommended clinical values.


## 1. Learning Objectives

After completing this notebook, you should be able to:

1. Distinguish the central compartment, peripheral compartment, elimination clearance, and intercompartmental clearance.
2. Explain why a rapid distribution phase and a slower terminal phase may appear after intravenous dosing.
3. Describe the main effects of $V_c$, $V_p$, $CL$, and $Q$ on concentration–time profiles.
4. Explain why $Q$ represents exchange between compartments rather than elimination from the body.


## 2. Why Is a Two-Compartment Model Needed?

In a one-compartment model, the body is represented as one immediately well-mixed space. After an intravenous bolus, concentration declines through a single exponential process:

$$
C(t)=C_0e^{-kt}
$$

In reality, a drug may first distribute to blood and rapidly equilibrating tissues and then gradually enter more slowly equilibrating tissues. Therefore, the decline after intravenous dosing may include two processes:

1. **Rapid early decline:** influenced by both distribution from the central to the peripheral compartment and elimination.
2. **Slower late decline:** observed after the compartments approach distribution equilibrium and determined by several model parameters.

A two-compartment model does not literally divide the body into two anatomical organs. It uses two mathematical spaces to approximate distribution processes occurring at different rates.

A useful interpretation is:

- **Central compartment:** plasma and tissues that equilibrate rapidly with blood.
- **Peripheral compartment:** tissues that exchange drug with the central compartment more slowly.
- **$Q$:** the capacity for exchange between the two compartments.
- **$CL$:** the capacity for irreversible elimination from the central compartment.

The value of the model is not simply that it contains more parameters. It teaches an important principle:

> A fall in plasma concentration does not always represent elimination; early decline may mainly reflect distribution.


## 3. Basic Structure of the Two-Compartment Model

Let $A_c$ be the amount of drug in the central compartment and $A_p$ the amount in the peripheral compartment:

$$
C_c=\frac{A_c}{V_c}
$$

$$
C_p=\frac{A_p}{V_p}
$$

After an intravenous bolus:

$$
\frac{dA_c}{dt}
=
-CL\frac{A_c}{V_c}
-Q\left(\frac{A_c}{V_c}-\frac{A_p}{V_p}\right)
$$

$$
\frac{dA_p}{dt}
=
Q\left(\frac{A_c}{V_c}-\frac{A_p}{V_p}\right)
$$

Parameter definitions:

| Symbol | Meaning | Common unit |
|---|---|---|
| $Dose$ | Intravenous bolus dose | mg |
| $V_c$ | Central volume | L |
| $V_p$ | Peripheral volume | L |
| $CL$ | Elimination clearance from the central compartment | L/h |
| $Q$ | Intercompartmental clearance | L/h |
| $A_c$, $A_p$ | Drug amounts in the two compartments | mg |
| $C_c$, $C_p$ | Drug concentrations in the two compartments | mg/L |

Immediately after dosing, the dose is initially in the central compartment:

$$
C_c(0)=\frac{Dose}{V_c}
$$

This explains why the initial central concentration after an intravenous bolus is mainly determined by dose and $V_c$.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

TRAPEZOID = getattr(np, "trapezoid", np.trapz)

from scipy.integrate import solve_ivp


def two_compartment_iv_bolus(t, dose_mg, vc_l, vp_l, cl_l_h, q_l_h):
    """Return central/peripheral amounts and concentrations after an IV bolus."""
    t = np.asarray(t, dtype=float)

    def rhs(_, amounts):
        a_c, a_p = amounts
        c_c = a_c / vc_l
        c_p = a_p / vp_l
        da_c = -cl_l_h * c_c - q_l_h * (c_c - c_p)
        da_p = q_l_h * (c_c - c_p)
        return [da_c, da_p]

    solution = solve_ivp(
        rhs,
        (float(t.min()), float(t.max())),
        [dose_mg, 0.0],
        t_eval=t,
        rtol=1e-8,
        atol=1e-10,
    )
    a_c, a_p = solution.y
    return a_c, a_p, a_c / vc_l, a_p / vp_l


def hybrid_constants(vc_l, vp_l, cl_l_h, q_l_h):
    """Calculate alpha, beta, and their half-lives for a two-compartment model."""
    k10 = cl_l_h / vc_l
    k12 = q_l_h / vc_l
    k21 = q_l_h / vp_l
    total = k10 + k12 + k21
    disc = max(total**2 - 4 * k10 * k21, 0.0)
    alpha = 0.5 * (total + np.sqrt(disc))
    beta = 0.5 * (total - np.sqrt(disc))
    return alpha, beta, np.log(2) / alpha, np.log(2) / beta


def one_compartment_iv_bolus(t, dose_mg, vd_l, cl_l_h):
    return dose_mg / vd_l * np.exp(-(cl_l_h / vd_l) * np.asarray(t))


## 4. Interactive Simulation 1: Central Compartment, Peripheral Compartment, and Drug Amount

The simulation displays:

- Central concentration
- Peripheral concentration
- Drug amount in each compartment
- Total amount remaining in the body
- Half-lives of the rapid and terminal phases

You can adjust:

- Dose
- Central volume, Vc
- Peripheral volume, Vp
- Elimination clearance, CL
- Intercompartmental clearance, Q
- Observation time
- Logarithmic y-axis

Change only one parameter at a time at first.

Focus on the following questions:

- Does central concentration reach its maximum immediately after dosing?
- Why does peripheral concentration start at zero and then increase?
- Does increasing $Q$ produce a faster early fall in central concentration?
- Do changes in $CL$ and $Q$ have the same effect on the total amount remaining in the body?


In [ ]:
def plot_two_compartment_profile(
    dose_mg=500,
    vc_l=15,
    vp_l=35,
    cl_l_h=5,
    q_l_h=8,
    time_h=24,
    log_scale=False,
):
    t = np.linspace(0, time_h, 700)
    a_c, a_p, c_c, c_p = two_compartment_iv_bolus(
        t, dose_mg, vc_l, vp_l, cl_l_h, q_l_h
    )
    alpha, beta, t_half_alpha, t_half_beta = hybrid_constants(
        vc_l, vp_l, cl_l_h, q_l_h
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, c_c, linewidth=2, label="Central concentration")
    ax.plot(t, c_p, linewidth=2, linestyle="--", label="Peripheral concentration")
    ax.set_title("Two-Compartment IV Bolus Model")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    if log_scale:
        ax.set_yscale("log")
        ax.set_ylim(bottom=max(np.min(c_c[c_c > 0]) * 0.7, 1e-3))
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t, a_c / dose_mg * 100, linewidth=2, label="Amount in central compartment")
    ax.plot(t, a_p / dose_mg * 100, linewidth=2, linestyle="--", label="Amount in peripheral compartment")
    ax.plot(t, (a_c + a_p) / dose_mg * 100, linewidth=2, linestyle=":", label="Amount remaining in body")
    ax.set_title("Drug Amount Across Compartments")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Percent of dose (%)")
    ax.set_ylim(0, 105)
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Dose", "Central volume (Vc)", "Peripheral volume (Vp)",
            "Elimination clearance (CL)", "Distribution clearance (Q)",
            "Alpha half-life", "Beta half-life", "Initial central concentration"
        ],
        "Value": [
            f"{dose_mg:.0f} mg", f"{vc_l:.1f} L", f"{vp_l:.1f} L",
            f"{cl_l_h:.1f} L/h", f"{q_l_h:.1f} L/h",
            f"{t_half_alpha:.2f} h", f"{t_half_beta:.2f} h",
            f"{dose_mg / vc_l:.2f} mg/L"
        ]
    })
    display(summary)


interact(
    plot_two_compartment_profile,
    dose_mg=FloatSlider(value=500, min=100, max=1500, step=50, description="Dose"),
    vc_l=FloatSlider(value=15, min=5, max=50, step=2.5, description="Vc"),
    vp_l=FloatSlider(value=35, min=5, max=100, step=5, description="Vp"),
    cl_l_h=FloatSlider(value=5, min=1, max=15, step=0.5, description="CL"),
    q_l_h=FloatSlider(value=8, min=0.5, max=30, step=0.5, description="Q"),
    time_h=Dropdown(options=[8, 12, 24, 48], value=24, description="Time"),
    log_scale=Checkbox(value=False, description="Log scale"),
);


## 5. Observation Tasks 1: Understanding Distribution

### Task A: Reference Parameters

Set:

- Dose = 500 mg
- Vc = 15 L
- Vp = 35 L
- CL = 5 L/h
- Q = 8 L/h
- Time = 24 h

Record:

- Initial central concentration
- Alpha half-life
- Beta half-life

Observe the directions of change in central and peripheral concentrations.

### Task B: Change Intercompartmental Clearance

Keep the other parameters unchanged and compare:

- Q = 2 L/h
- Q = 8 L/h
- Q = 24 L/h

Observe:

- Does central concentration fall more rapidly at early times as Q increases?
- Does peripheral concentration rise more rapidly?
- Is a larger Q equivalent to faster elimination from the body?

### Task C: Change Central Volume

Increase Vc from 15 L to 30 L.

Observe:

- How does the initial central concentration change?
- Why does $C_c(0)=Dose/V_c$?
- Does the early profile change markedly?

### Task D: Compare CL and Q

Increase CL alone and then increase Q alone.

Observe:

- Does increasing CL remove total drug from the body more rapidly?
- Does increasing Q mainly redistribute drug between compartments?


## 6. Distribution Phase, Terminal Phase, and Hybrid Rate Constants

After an intravenous bolus, central concentration in a two-compartment model is often written as:

$$
C_c(t)=Ae^{-\alpha t}+Be^{-\beta t}
$$

where:

$$
\alpha>\beta
$$

The two phases may be interpreted as:

- **$\alpha$ phase:** a faster early decline in which distribution is prominent.
- **$\beta$ phase:** a slower terminal decline.

The corresponding half-lives are:

$$
t_{1/2,\alpha}=\frac{0.693}{\alpha}
$$

$$
t_{1/2,\beta}=\frac{0.693}{\beta}
$$

Important points:

- $\alpha$ and $\beta$ are hybrid rate constants jointly determined by $V_c$, $V_p$, $CL$, and $Q$.
- $\alpha$ is not simply equal to $Q$.
- $\beta$ is not simply equal to $CL/V$.
- A longer terminal half-life does not necessarily indicate lower clearance alone; distribution volumes and intercompartmental exchange may also contribute.

A single half-life cannot fully describe the entire process in a two-compartment model.


## 7. Interactive Simulation 2: Effect of Intercompartmental Clearance $Q$

This simulation compares lower, reference, and higher values of $Q$.

To focus on distribution, examine the early central concentration rather than only the last few time points.

Consider:

- When $Q$ is low, does the drug enter the peripheral compartment more slowly?
- When $Q$ is high, does central concentration fall more rapidly at early times?
- Do the curves become more similar at later times?
- Do the alpha and beta half-lives change in the same proportion when Q changes?


In [ ]:
def plot_distribution_clearance_comparison(
    dose_mg=500,
    vc_l=15,
    vp_l=35,
    cl_l_h=5,
    q_l_h=8,
    time_h=12,
):
    t = np.linspace(0, time_h, 600)
    q_values = [max(q_l_h / 3, 0.1), q_l_h, q_l_h * 3]
    labels = ["Lower Q", "Reference Q", "Higher Q"]

    fig, ax = plt.subplots(figsize=(9, 5))
    rows = []
    for q_value, label in zip(q_values, labels):
        _, _, c_c, _ = two_compartment_iv_bolus(
            t, dose_mg, vc_l, vp_l, cl_l_h, q_value
        )
        alpha, beta, h_alpha, h_beta = hybrid_constants(vc_l, vp_l, cl_l_h, q_value)
        ax.plot(t, c_c, linewidth=2, label=f"{label}: Q={q_value:.1f} L/h")
        rows.append({
            "Scenario": label,
            "Q_L_h": q_value,
            "Alpha_half_life_h": h_alpha,
            "Beta_half_life_h": h_beta,
            "Concentration_at_1h_mg_L": np.interp(1, t, c_c),
        })

    ax.set_title("Effect of Distribution Clearance on Central Concentration")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Central concentration (mg/L)")
    ax.legend()
    plt.show()
    display(pd.DataFrame(rows).round(3))


interact(
    plot_distribution_clearance_comparison,
    dose_mg=FloatSlider(value=500, min=100, max=1500, step=50, description="Dose"),
    vc_l=FloatSlider(value=15, min=5, max=50, step=2.5, description="Vc"),
    vp_l=FloatSlider(value=35, min=5, max=100, step=5, description="Vp"),
    cl_l_h=FloatSlider(value=5, min=1, max=15, step=0.5, description="CL"),
    q_l_h=FloatSlider(value=8, min=1, max=20, step=1, description="Q"),
    time_h=Dropdown(options=[6, 12, 24], value=12, description="Time"),
);


## 8. Observation Tasks 2: Distribution Is Not Elimination

### Task A: Reference Comparison

Set:

- Dose = 500 mg
- Vc = 15 L
- Vp = 35 L
- CL = 5 L/h
- Q = 8 L/h
- Time = 12 h

For each scenario, record:

- Concentration at 1 h
- Alpha half-life
- Beta half-life

### Task B: Increase Q

Examine the Higher Q curve.

Consider:

- Why is central concentration lower at 1 hour?
- Has the drug left the body, or has more drug entered the peripheral compartment?
- If only central concentration is measured, could distribution be mistaken for elimination?

### Task C: Reduce CL

Keep Q unchanged and reduce CL.

Observe:

- Does the terminal phase become slower?
- Does drug remain in the body for longer?
- How is this different from changing Q alone?


## 11. Scope and Limitations

1. A two-compartment model is a mathematical approximation; the central and peripheral compartments are not fixed anatomical organs.
2. This module includes only intravenous bolus dosing and linear elimination. Oral absorption, infusion, nonlinear elimination, and time-varying parameters are not included.
3. The example parameters are educational and are not validated for a particular drug or population.
4. No parameter estimation, model comparison, residual analysis, or external validation is performed.
5. Real model selection requires appropriate sampling, data quality assessment, diagnostic plots, prior pharmacological knowledge, and a clearly defined purpose.


## 12. Self-Assessment: Two-Compartment Model and Distribution

Answer the questions before viewing the next cell.

---

### Question 1

Which two factors most directly determine the initial central concentration after an intravenous bolus?

A. $Dose$ and $V_c$  
B. $Q$ and $V_p$  
C. $CL$ and MIC  
D. $T_{max}$ and bioavailability  

---

### Question 2

What does an increase in intercompartmental clearance $Q$ most directly indicate?

A. Increased renal elimination  
B. Faster exchange between the central and peripheral compartments  
C. Faster oral absorption  
D. Lower MIC  





## 13. Self-Assessment Answers

### Question 1

**Answer: A**

Immediately after an intravenous bolus:

$$
C_c(0)=\frac{Dose}{V_c}
$$

---

### Question 2

**Answer: B**

$Q$ describes bidirectional exchange between compartments. It does not represent irreversible elimination from the body.



## 14. Notebook Summary

This notebook used an intravenous bolus two-compartment model to distinguish distribution from elimination.

Key conclusions are:

1. A two-compartment model uses a central compartment, a peripheral compartment, $CL$, and $Q$ to describe distribution and elimination.
2. Early concentration decline after intravenous dosing may mainly reflect distribution rather than irreversible elimination.
3. $Q$ controls exchange between compartments, whereas $CL$ controls elimination from the body.
4. The rapid and terminal phases are hybrid processes determined by several parameters rather than one physiological process.
5. Both early and late samples are required to identify distribution, the terminal phase, and a reasonable model structure.

The link to the later NCA notebook can be summarized as:

$$
Understanding\ model\ structure
\rightarrow
Distinguishing\ distribution\ and\ terminal\ phases
\rightarrow
Choosing\ sampling\ times
\rightarrow
Interpreting\ terminal\ half\text{-}life
$$


## 15. References

1. Mould DR, Upton RN. Basic concepts in population modeling, simulation, and model-based drug development. *CPT Pharmacometrics Syst Pharmacol.* 2012;1:e6.  
2. Toutain PL, Bousquet-Mélou A. Volumes of distribution. *J Vet Pharmacol Ther.* 2004;27:441–453.  
3. Rowland M, Tozer TN. *Clinical Pharmacokinetics and Pharmacodynamics: Concepts and Applications.*  
4. Gabrielsson J, Weiner D. *Pharmacokinetic and Pharmacodynamic Data Analysis: Concepts and Applications.*
